In [ ]:
import astropy.units as u
import astropy.constants as const
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from tqdm.notebook import tqdm
import json
import h5py
import os

import warnings
warnings.filterwarnings("ignore")

from DiscEvolution.grid import Grid
from DiscEvolution.star import SimpleStar
from DiscEvolution.eos import LocallyIsothermalEOS  
from DiscEvolution.eos import SimpleDiscEOS
from DiscEvolution.eos import IrradiatedEOS  
from DiscEvolution.disc import AccretionDisc
from DiscEvolution.viscous_evolution import ViscousEvolutionFV
from DiscEvolution.driver import PlanetDiscDriver
from DiscEvolution.io import Event_Controller
from DiscEvolution.dust import DustGrowthTwoPop, SingleFluidDrift
from DiscEvolution.viscous_evolution import HybridWindModel
from DiscEvolution.opacity import Zhu2012
from DiscEvolution.constants import yr as yr_code, GasConst, Omega0, AU, Msun  # DiscEvolution uses code units: 1 year = 2π code time units
from DiscEvolution.chemistry import SimpleCOAtomAbund, EquilibriumCOChemOberg
from DiscEvolution import planet_formation as pf
from DiscEvolution.diffusion import TracerDiffusion 
from DiscEvolution.planet_formation import Planets, Bitsch2015Model
from DiscEvolution.constants import *

# Files

In [ ]:
def h5_to_dict(h5obj):
    """Recursively load HDF5 file into nested dict."""
    out = {}
    for k, v in h5obj.items():
        if isinstance(v, h5py.Dataset):
            out[k] = v[()]  # load dataset into memory
        else:
            out[k] = h5_to_dict(v)  # recurse into group
    return out

# Define file paths with descriptive labels (not CR/XR/CX)
file_paths = {
    "1Myr_NoDZ_1e-8_0.1Msol_50Rd_1e-5" : "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/GrowthTracksAfterMerge/output/noDZ_alpha1e-5/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260928_132035.h5",
    "1Myr_DZ_1e-8_0.1Msol_50Rd_1e-5_1e-3": "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/GrowthTracksAfterMerge/output/DZ/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260928_132540.h5",
    "3Myr_NoDZ_1e-8_0.1Msol_50Rd_1e-5" : "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/GrowthTracksAfterMerge/output/noDZ_alpha1e-5/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260928_223919.h5",
    "3Myr_DZ_1e-8_0.1Msol_50Rd_1e-5_1e-3" : "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/GrowthTracksAfterMerge/output/DZ/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260928_223939.h5" ,
    "1Myr_DZ_premerge" : "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/MiniPopSynth/output/CR_1.0Myr/1e-8_0.1Msol_50Rd/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260917_205653.h5", 
    "1Myr_NoDZ_premerge_1e-3" : "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/MiniGrowthTracks/output/WindOnly/CR_1Myr_1e-8_0.1Msol_50Rd/alpha_1e-5/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260924_132423.h5" ,
    # "1Myr_noDZ_premerge_1e-5": "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/MiniGrowthTracks/output/WindOnly/CR_1Myr_1e-8_0.1Msol_50Rd/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260923_120633.h5",
    "0.1Myr_DZ_1e-8_0.1Msol_50Rd_1e-5_1e-3" : "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/GrowthTracksAfterMerge/output/DZ/0.1Myr/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20260929_160821.h5" ,
    "0.1Myr_NoDZ_1e-8_0.1Msol_50Rd_1e-5" : "/Users/james/Documents/GitHub/DiscEvolution/james/september2026/GrowthTracksAfterMerge/output/noDZ_alpha1e-5/0.1Myr/winds_mig_psi2.533e+01_Mdot1.0e-08_M1.0e-01_Rd5.0e+01_20261002_103136.h5"
}

# Load all files into a single nested dictionary
datasets = {}
for label, path in file_paths.items():
    with h5py.File(path, "r") as f:
        datasets[label] = {
            "data": h5_to_dict(f),
            "attrs": dict(f.attrs),
        }

print(f"✓ Loaded {len(datasets)} simulations")
for label in datasets.keys():
    print(f"  {label}: {list(datasets[label]['data'].keys())}")

# After Merge: No Deadzone alpha=1e-5 

In [ ]:
dat = datasets["3Myr_NoDZ_1e-8_0.1Msol_50Rd_1e-5"]["data"]
# for item in dat.keys():
#    print(item)

In [ ]:
R = dat["R"]
L = dat["Lambda"]
t = dat["t"]
Rdz = dat['R_dz_t']
psi = dat['psi_R']
alpha = dat['alpha_R']

# Planets stuff
Rp = dat['Rp']      # dict of planets radii
Mcs = dat['Mcs']      # dict of planets cores
Mes = dat['Mes']      # dict of planets enveloppes
Mtot = { planet_id: Mcs[planet_id] + Mes[planet_id] for planet_id in Mcs }   # dict of planets total masses
X_cores = dat["X_cores"]
X_envs = dat["X_envs"]

# Disc Stuff 
Sigma_g = dat['Sigma_G']
Sigma_d = dat['Sigma_dust']
Sigma_peb = dat['Sigma_pebbles']
Sigma_pmal = dat['Sigma_planetesimals']
tsnap = dat['time_snap']
St_dust = dat['St'][:, 0, :]
St_peb = dat['St'][:, 1, :]
St_pmal = dat['St'][:, 2, :]
peb_size = dat["Sigma_pebble_size"]  # shape: (snapshots, radii), cm

planet_ids = sorted(Rp.keys(), key=int)

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML

planet_ids = sorted(Rp.keys(), key=int)

# Ensure all planet tracks share a common length
nframes_data = min(
    len(t),
    *(len(Rp[pid]) for pid in planet_ids),
    *(len(Mtot[pid]) for pid in planet_ids),
    *(len(Mes[pid]) for pid in planet_ids),
)

time_years = np.asarray(t[:nframes_data])
time_myr = time_years / 1e6

# Select snapshots approximately every 0.05 Myr
target_times = np.arange(0, time_myr[-1] + 0.05, 0.05)
frame_indices = np.searchsorted(time_myr, target_times)
frame_indices = np.clip(frame_indices, 0, nframes_data - 1)
frame_indices = np.unique(frame_indices)

all_radii = np.concatenate([
    np.asarray(Rp[pid][:nframes_data]) for pid in planet_ids
])
all_masses = np.concatenate([
    np.asarray(Mtot[pid][:nframes_data]) for pid in planet_ids
])

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(np.nanmin(all_radii) * 0.8, np.nanmax(all_radii) * 1.2)
ax.set_ylim(np.nanmin(all_masses) * 0.8, np.nanmax(all_masses) * 1.2)

ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Total mass")
ax.grid(True, which="both", alpha=0.2)

norm = Normalize(vmin=0.0, vmax=1.0)
cmap = plt.cm.viridis

scatter = ax.scatter(
    [],
    [],
    c=[],
    cmap=cmap,
    norm=norm,
    s=35,
    alpha=0.7,
)

# deadzone_line = ax.axvline(
#     deadzone[0],
#     ymin=0,
#     ymax=0.10,
#     color="red",
#     linestyle=":",
#     linewidth=1.5,
# )

colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label(r"$M_{\rm env}/M_{\rm tot}$")

def update(frame):
    current_index = frame_indices[frame]

    trail_distances = []
    trail_masses = []
    trail_ratios = []

    for planet_id in planet_ids:
        distance = np.asarray(Rp[planet_id][:current_index + 1])
        total_mass = np.asarray(Mtot[planet_id][:current_index + 1])
        envelope_mass = np.asarray(Mes[planet_id][:current_index + 1])

        total_mass = np.maximum(total_mass, 1e-300)
        envelope_fraction = envelope_mass / total_mass

        trail_distances.append(distance)
        trail_masses.append(total_mass)
        trail_ratios.append(envelope_fraction)

    scatter.set_offsets(np.column_stack((
        np.concatenate(trail_distances),
        np.concatenate(trail_masses),
    )))
    scatter.set_array(np.concatenate(trail_ratios))

    # deadzone_line.set_xdata([deadzone[current_index], deadzone[current_index]])

    ax.set_title(
        rf"$t={time_myr[current_index]:.3f}$ Myr"
    )

    return scatter,

animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_indices),
    interval=150,
    blit=False,
)


from matplotlib.animation import PillowWriter
# animation.save("/gifs/NoDZ_1e-5_aftermerge.gif", writer=PillowWriter(fps=8), dpi=300)

plt.close(fig)
HTML(animation.to_jshtml())

## Sigma_G

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_g))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_G (g cm^-2)')
#ax.set_xlim(8,10)
# ax.set_ylim(1e-10,1e7)

for i in range(len(Sigma_g)):
    ax.loglog(
        R, Sigma_g[i],
        color="red", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma_d[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    # rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    # ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    # ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_g))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:2]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_g[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_g))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_g)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_g$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_g[frame_indices[0]], color="red", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_g[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma_D

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_d))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_d)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e3)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_d$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_d[frame_indices[0]], color="red", lw=1.2, ls = "--")
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_d[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma Peb

In [ ]:
Sigma_bs = Sigma_peb

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_bs))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_bs)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_bs$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_bs[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_bs[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Stokes Number

In [ ]:
St = St_peb

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Stokes Number")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Pebble Size

In [ ]:
St = peb_size

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Pebble Size (cm)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## psi and alpha

In [ ]:
St = psi

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Pebble Size (cm)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

In [ ]:
print(psi)

In [ ]:
St = alpha

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Pebble Size (cm)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

In [ ]:
print(alpha)

# After Merge: No Deadzone alpha = 1e-5 (0.1Myr only)

In [ ]:
dat = datasets["0.1Myr_NoDZ_1e-8_0.1Msol_50Rd_1e-5"]["data"]
# for item in dat.keys():
#    print(item)

In [ ]:
R = dat["R"]
L = dat["Lambda"]
t = dat["t"]
Rdz = dat['R_dz_t']
psi = dat['psi_R']
alpha = dat['alpha_R']

# Planets stuff
Rp = dat['Rp']      # dict of planets radii
Mcs = dat['Mcs']      # dict of planets cores
Mes = dat['Mes']      # dict of planets enveloppes
Mtot = { planet_id: Mcs[planet_id] + Mes[planet_id] for planet_id in Mcs }   # dict of planets total masses
X_cores = dat["X_cores"]
X_envs = dat["X_envs"]

# Disc Stuff 
Sigma_g = dat['Sigma_G']
Sigma_d = dat['Sigma_dust']
Sigma_peb = dat['Sigma_pebbles']
Sigma_pmal = dat['Sigma_planetesimals']
tsnap = dat['time_snap']
St_dust = dat['St'][:, 0, :]
St_peb = dat['St'][:, 1, :]
St_pmal = dat['St'][:, 2, :]
peb_size = dat["Sigma_pebble_size"]  # shape: (snapshots, radii), cm

planet_ids = sorted(Rp.keys(), key=int)

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML

planet_ids = sorted(Rp.keys(), key=int)

# Ensure all planet tracks share a common length
nframes_data = min(
    len(t),
    *(len(Rp[pid]) for pid in planet_ids),
    *(len(Mtot[pid]) for pid in planet_ids),
    *(len(Mes[pid]) for pid in planet_ids),
)

time_years = np.asarray(t[:nframes_data])
time_myr = time_years / 1e6

# Select snapshots approximately every 0.01 Myr
target_times = np.arange(0, time_myr[-1] + 0.01, 0.01)
frame_indices = np.searchsorted(time_myr, target_times)
frame_indices = np.clip(frame_indices, 0, nframes_data - 1)
frame_indices = np.unique(frame_indices)

all_radii = np.concatenate([
    np.asarray(Rp[pid][:nframes_data]) for pid in planet_ids
])
all_masses = np.concatenate([
    np.asarray(Mtot[pid][:nframes_data]) for pid in planet_ids
])

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(np.nanmin(all_radii) * 0.8, np.nanmax(all_radii) * 1.2)
ax.set_ylim(np.nanmin(all_masses) * 0.8, np.nanmax(all_masses) * 1.2)

ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Total mass")
ax.grid(True, which="both", alpha=0.2)

norm = Normalize(vmin=0.0, vmax=1.0)
cmap = plt.cm.viridis

scatter = ax.scatter(
    [],
    [],
    c=[],
    cmap=cmap,
    norm=norm,
    s=35,
    alpha=0.7,
)

# deadzone_line = ax.axvline(
#     deadzone[0],
#     ymin=0,
#     ymax=0.10,
#     color="red",
#     linestyle=":",
#     linewidth=1.5,
# )

colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label(r"$M_{\rm env}/M_{\rm tot}$")

def update(frame):
    current_index = frame_indices[frame]

    trail_distances = []
    trail_masses = []
    trail_ratios = []

    for planet_id in planet_ids:
        distance = np.asarray(Rp[planet_id][:current_index + 1])
        total_mass = np.asarray(Mtot[planet_id][:current_index + 1])
        envelope_mass = np.asarray(Mes[planet_id][:current_index + 1])

        total_mass = np.maximum(total_mass, 1e-300)
        envelope_fraction = envelope_mass / total_mass

        trail_distances.append(distance)
        trail_masses.append(total_mass)
        trail_ratios.append(envelope_fraction)

    scatter.set_offsets(np.column_stack((
        np.concatenate(trail_distances),
        np.concatenate(trail_masses),
    )))
    scatter.set_array(np.concatenate(trail_ratios))

    # deadzone_line.set_xdata([deadzone[current_index], deadzone[current_index]])

    ax.set_title(
        rf"$t={time_myr[current_index]:.3f}$ Myr"
    )

    return scatter,

animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_indices),
    interval=150,
    blit=False,
)


from matplotlib.animation import PillowWriter
# animation.save("/gifs/NoDZ_1e-5_aftermerge.gif", writer=PillowWriter(fps=8), dpi=300)

plt.close(fig)
HTML(animation.to_jshtml())

## Sigma_G

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_g))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_g)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 2.2)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_g$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_g[frame_indices[0]], color="red", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_g[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma_D

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_d))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_d)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e3)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_d$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_d[frame_indices[0]], color="red", lw=1.2, ls = "--")
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_d[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma peb

In [ ]:
Sigma_bs = Sigma_peb

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_bs))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_bs)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_bs$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_bs[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_bs[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Stokes

In [ ]:
St = St_peb

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Stokes Number")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Pebble Size

In [ ]:
St = peb_size

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Pebble Size (cm)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## psi and alpha

In [ ]:
St = psi

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Pebble Size (cm)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

In [ ]:
print(psi)

In [ ]:
St = alpha

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Pebble Size (cm)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

In [ ]:
print(alpha)

# After Merge: Deadzone alpha=1e-5 -> 1e-3

In [ ]:
dat = datasets["3Myr_DZ_1e-8_0.1Msol_50Rd_1e-5_1e-3"]["data"]
for item in dat.keys():
   print(item)

In [ ]:
R = dat["R"]
L = dat["Lambda"]
t = dat["t"]
Rdz = dat['R_dz_t']
psi = dat['psi_R']

# Planets stuff
Rp = dat['Rp']      # dict of planets radii
Mcs = dat['Mcs']      # dict of planets cores
Mes = dat['Mes']      # dict of planets enveloppes
Mtot = { planet_id: Mcs[planet_id] + Mes[planet_id] for planet_id in Mcs }   # dict of planets total masses
X_cores = dat["X_cores"]
X_envs = dat["X_envs"]

# Disc Stuff 
Sigma_g = dat['Sigma_G']
Sigma_d = dat['Sigma_dust']
Sigma_peb = dat['Sigma_pebbles']
Sigma_pmal = dat['Sigma_planetesimals']
tsnap = dat['time_snap']

planet_ids = sorted(Rp.keys(), key=int)

In [ ]:
Rp['1']

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML

planet_ids = sorted(Rp.keys(), key=int)

# Ensure all planet tracks share a common length
nframes_data = min(
    len(t),
    *(len(Rp[pid]) for pid in planet_ids),
    *(len(Mtot[pid]) for pid in planet_ids),
    *(len(Mes[pid]) for pid in planet_ids),
)

time_years = np.asarray(t[:nframes_data])
time_myr = time_years / 1e6
deadzone = np.asarray(Rdz[:nframes_data])

# Select snapshots approximately every 0.05 Myr
target_times = np.arange(0, time_myr[-1] + 0.05, 0.05)
frame_indices = np.searchsorted(time_myr, target_times)
frame_indices = np.clip(frame_indices, 0, nframes_data - 1)
frame_indices = np.unique(frame_indices)

all_radii = np.concatenate([
    np.asarray(Rp[pid][:nframes_data]) for pid in planet_ids
])
all_masses = np.concatenate([
    np.asarray(Mtot[pid][:nframes_data]) for pid in planet_ids
])

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(np.nanmin(all_radii) * 0.8, np.nanmax(all_radii) * 1.2)
ax.set_ylim(np.nanmin(all_masses) * 0.8, np.nanmax(all_masses) * 1.2)

ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Total mass")
ax.grid(True, which="both", alpha=0.2)

norm = Normalize(vmin=0.0, vmax=1.0)
cmap = plt.cm.viridis

scatter = ax.scatter(
    [],
    [],
    c=[],
    cmap=cmap,
    norm=norm,
    s=35,
    alpha=0.7,
)

deadzone_line = ax.axvline(
    deadzone[0],
    ymin=0,
    ymax=0.10,
    color="red",
    linestyle=":",
    linewidth=1.5,
)

colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label(r"$M_{\rm env}/M_{\rm tot}$")

def update(frame):
    current_index = frame_indices[frame]

    trail_distances = []
    trail_masses = []
    trail_ratios = []

    for planet_id in planet_ids:
        distance = np.asarray(Rp[planet_id][:current_index + 1])
        total_mass = np.asarray(Mtot[planet_id][:current_index + 1])
        envelope_mass = np.asarray(Mes[planet_id][:current_index + 1])

        total_mass = np.maximum(total_mass, 1e-300)
        envelope_fraction = envelope_mass / total_mass

        trail_distances.append(distance)
        trail_masses.append(total_mass)
        trail_ratios.append(envelope_fraction)

    scatter.set_offsets(np.column_stack((
        np.concatenate(trail_distances),
        np.concatenate(trail_masses),
    )))
    scatter.set_array(np.concatenate(trail_ratios))

    deadzone_line.set_xdata([deadzone[current_index], deadzone[current_index]])

    ax.set_title(
        rf"$t={time_myr[current_index]:.3f}$ Myr"
    )

    return scatter,

animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_indices),
    interval=150,
    blit=False,
)


from matplotlib.animation import PillowWriter
# animation.save("/gifs/NoDZ_1e-5_aftermerge.gif", writer=PillowWriter(fps=8), dpi=300)

plt.close(fig)
HTML(animation.to_jshtml())

## Sigma_G

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_g))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_G (g cm^-2)')
#ax.set_xlim(8,10)
# ax.set_ylim(1e-10,1e7)

for i in range(len(Sigma_g)):
    ax.loglog(
        R, Sigma_g[i],
        color="red", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma_d[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_g))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:2]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_g[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_g))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_g)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_g$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_g[frame_indices[0]], color="red", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_g[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma D

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_d))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_D (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e3)

for i in range(len(Sigma_d)):
    ax.loglog(
        R, Sigma_d[i],
        color="red", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = "--"
    )
    # ax.loglog(
    #     R, Sigma_d[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_d))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_d[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_d))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_d)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e3)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_d$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_d[frame_indices[0]], color="red", lw=1.2, ls = "--")
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_d[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma total

In [ ]:
Sigma = Sigma_g + Sigma_d + Sigma_peb + Sigma_pmal

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(Sigma)):
    ax.loglog(
        R, Sigma[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

# After Merge: Deadzone alpha=1e-5 -> 1e-3 (0.1 Myr only)

In [ ]:
dat = datasets["0.1Myr_DZ_1e-8_0.1Msol_50Rd_1e-5_1e-3"]["data"]
for item in dat.keys():
   print(item)

In [ ]:
R = dat["R"]
L = dat["Lambda"]
t = dat["t"]
Rdz = dat['R_dz_t']
psi = dat['psi_R']

# Planets stuff
Rp = dat['Rp']      # dict of planets radii
Mcs = dat['Mcs']      # dict of planets cores
Mes = dat['Mes']      # dict of planets enveloppes
Mtot = { planet_id: Mcs[planet_id] + Mes[planet_id] for planet_id in Mcs }   # dict of planets total masses
X_cores = dat["X_cores"]
X_envs = dat["X_envs"]

# Disc Stuff 
Sigma_g = dat['Sigma_G']
Sigma_d = dat['Sigma_dust']
Sigma_peb = dat['Sigma_pebbles']
Sigma_pmal = dat['Sigma_planetesimals']
tsnap = dat['time_snap']
St_dust = dat['St'][:, 0, :]
St_peb = dat['St'][:, 1, :]
St_pmal = dat['St'][:, 2, :]
peb_size = dat["Sigma_pebble_size"]  # shape: (snapshots, radii), cm

planet_ids = sorted(Rp.keys(), key=int)

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML

planet_ids = sorted(Rp.keys(), key=int)

# Ensure all planet tracks share a common length
nframes_data = min(
    len(t),
    *(len(Rp[pid]) for pid in planet_ids),
    *(len(Mtot[pid]) for pid in planet_ids),
    *(len(Mes[pid]) for pid in planet_ids),
)

time_years = np.asarray(t[:nframes_data])
time_myr = time_years / 1e6
deadzone = np.asarray(Rdz[:nframes_data])

# Select snapshots approximately every 0.01 Myr
target_times = np.arange(0, time_myr[-1] + 0.005, 0.005)
frame_indices = np.searchsorted(time_myr, target_times)
frame_indices = np.clip(frame_indices, 0, nframes_data - 1)
frame_indices = np.unique(frame_indices)

all_radii = np.concatenate([
    np.asarray(Rp[pid][:nframes_data]) for pid in planet_ids
])
all_masses = np.concatenate([
    np.asarray(Mtot[pid][:nframes_data]) for pid in planet_ids
])

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(np.nanmin(all_radii) * 0.8, np.nanmax(all_radii) * 1.2)
ax.set_ylim(np.nanmin(all_masses) * 0.8, np.nanmax(all_masses) * 1.2)

ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Total mass")
ax.grid(True, which="both", alpha=0.2)

norm = Normalize(vmin=0.0, vmax=1.0)
cmap = plt.cm.viridis

scatter = ax.scatter(
    [],
    [],
    c=[],
    cmap=cmap,
    norm=norm,
    s=35,
    alpha=0.7,
)

deadzone_line = ax.axvline(
    deadzone[0],
    ymin=0,
    ymax=0.10,
    color="red",
    linestyle=":",
    linewidth=1.5,
)

colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label(r"$M_{\rm env}/M_{\rm tot}$")

def update(frame):
    current_index = frame_indices[frame]

    trail_distances = []
    trail_masses = []
    trail_ratios = []

    for planet_id in planet_ids:
        distance = np.asarray(Rp[planet_id][:current_index + 1])
        total_mass = np.asarray(Mtot[planet_id][:current_index + 1])
        envelope_mass = np.asarray(Mes[planet_id][:current_index + 1])

        total_mass = np.maximum(total_mass, 1e-300)
        envelope_fraction = envelope_mass / total_mass

        trail_distances.append(distance)
        trail_masses.append(total_mass)
        trail_ratios.append(envelope_fraction)

    scatter.set_offsets(np.column_stack((
        np.concatenate(trail_distances),
        np.concatenate(trail_masses),
    )))
    scatter.set_array(np.concatenate(trail_ratios))

    deadzone_line.set_xdata([deadzone[current_index], deadzone[current_index]])

    ax.set_title(
        rf"$t={time_myr[current_index]:.3f}$ Myr"
    )

    return scatter,

animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_indices),
    interval=150,
    blit=False,
)


from matplotlib.animation import PillowWriter
# animation.save("/gifs/NoDZ_1e-5_aftermerge.gif", writer=PillowWriter(fps=8), dpi=300)

plt.close(fig)
HTML(animation.to_jshtml())

## Sigma G

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_g))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_G (g cm^-2)')
#ax.set_xlim(8,10)
# ax.set_ylim(1e-10,1e7)

for i in range(len(Sigma_g)):
    ax.loglog(
        R, Sigma_g[i],
        color="red", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma_d[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_g))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:2]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_g[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_g))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_g)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_g$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_g[frame_indices[0]], color="red", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_g[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma D

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_d))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_D (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e3)

for i in range(len(Sigma_d)):
    ax.loglog(
        R, Sigma_d[i],
        color="red", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = "--"
    )
    # ax.loglog(
    #     R, Sigma_d[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_d))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_d[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_d))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_d)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e3)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_d$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_d[frame_indices[0]], color="red", lw=1.2, ls = "--")
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_d[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma big solids

In [ ]:
Sigma_bs = Sigma_peb

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_bs))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_bs (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(Sigma_bs)):
    ax.loglog(
        R, Sigma_bs[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma_bs[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_bs))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_bs[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_bs))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_bs)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_bs$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_bs[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_bs[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma Total

In [ ]:
Sigma = Sigma_g + Sigma_d + Sigma_peb + Sigma_pmal

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(Sigma)):
    ax.loglog(
        R, Sigma[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Stokes number

In [ ]:
St = St_peb

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(St))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Stokes Number')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(St)):
    ax.loglog(
        R, St[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, St[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(St))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(St[:n_snap])

    St_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, St_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Stokes Number")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Pebble size

In [ ]:
St = peb_size

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(St))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Pebble size (cm)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(St)):
    ax.loglog(
        R, St[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, St[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(St))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(St[:n_snap])

    St_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, St_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Pebble Size (cm)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

# Pre Merge: Deadzone alpha=1e-5 -> 1e-3

In [ ]:
dat = datasets["1Myr_DZ_premerge"]["data"]
for item in dat.keys():
   print(item)

In [ ]:
R = dat["R"]
L = dat["Lambda"]
t = dat["t"]
Rdz = dat['R_dz_t']
psi = dat['psi_R']

# Planets stuff
Rp = dat['Rp']      # dict of planets radii
Mcs = dat['Mcs']      # dict of planets cores
Mes = dat['Mes']      # dict of planets enveloppes
Mtot = { planet_id: Mcs[planet_id] + Mes[planet_id] for planet_id in Mcs }   # dict of planets total masses
X_cores = dat["X_cores"]
X_envs = dat["X_envs"]

# Disc Stuff 
Sigma_g = dat['Sigma_G']
Sigma_d = dat['Sigma_dust']
Sigma_peb = dat['Sigma_pebbles']
Sigma_pmal = dat['Sigma_planetesimals']
tsnap = dat['time_snap']
St_dust = dat['St'][:, 0, :]
St_peb = dat['St'][:, 1, :]
St_pmal = dat['St'][:, 2, :]

planet_ids = sorted(Rp.keys(), key=int)
planet_ids

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML

planet_ids = sorted(Rp.keys(), key=int)

# Ensure all planet tracks share a common length
nframes_data = min(
    len(t),
    *(len(Rp[pid]) for pid in planet_ids),
    *(len(Mtot[pid]) for pid in planet_ids),
    *(len(Mes[pid]) for pid in planet_ids),
)

time_years = np.asarray(t[:nframes_data])
time_myr = time_years / 1e6
deadzone = np.asarray(Rdz[:nframes_data])

# Select snapshots approximately every 0.01 Myr
target_times = np.arange(0, time_myr[-1] + 0.01, 0.01)
frame_indices = np.searchsorted(time_myr, target_times)
frame_indices = np.clip(frame_indices, 0, nframes_data - 1)
frame_indices = np.unique(frame_indices)

all_radii = np.concatenate([
    np.asarray(Rp[pid][:nframes_data]) for pid in planet_ids
])
all_masses = np.concatenate([
    np.asarray(Mtot[pid][:nframes_data]) for pid in planet_ids
])

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(np.nanmin(all_radii) * 0.8, np.nanmax(all_radii) * 1.2)
ax.set_ylim(np.nanmin(all_masses) * 0.8, np.nanmax(all_masses) * 1.2)

ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Total mass")
ax.grid(True, which="both", alpha=0.2)

norm = Normalize(vmin=0.0, vmax=1.0)
cmap = plt.cm.viridis

scatter = ax.scatter(
    [],
    [],
    c=[],
    cmap=cmap,
    norm=norm,
    s=35,
    alpha=0.7,
)

deadzone_line = ax.axvline(
    deadzone[0],
    ymin=0,
    ymax=0.10,
    color="red",
    linestyle=":",
    linewidth=1.5,
)

colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label(r"$M_{\rm env}/M_{\rm tot}$")

def update(frame):
    current_index = frame_indices[frame]

    trail_distances = []
    trail_masses = []
    trail_ratios = []

    for planet_id in planet_ids:
        distance = np.asarray(Rp[planet_id][:current_index + 1])
        total_mass = np.asarray(Mtot[planet_id][:current_index + 1])
        envelope_mass = np.asarray(Mes[planet_id][:current_index + 1])

        total_mass = np.maximum(total_mass, 1e-300)
        envelope_fraction = envelope_mass / total_mass

        trail_distances.append(distance)
        trail_masses.append(total_mass)
        trail_ratios.append(envelope_fraction)

    scatter.set_offsets(np.column_stack((
        np.concatenate(trail_distances),
        np.concatenate(trail_masses),
    )))
    scatter.set_array(np.concatenate(trail_ratios))

    deadzone_line.set_xdata([deadzone[current_index], deadzone[current_index]])

    ax.set_title(
        rf"$t={time_myr[current_index]:.3f}$ Myr"
    )

    return scatter,

animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_indices),
    interval=150,
    blit=False,
)


from matplotlib.animation import PillowWriter
# animation.save("/gifs/NoDZ_1e-5_aftermerge.gif", writer=PillowWriter(fps=8), dpi=300)

plt.close(fig)
HTML(animation.to_jshtml())

## Sigma G

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_g))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_G (g cm^-2)')
#ax.set_xlim(8,10)
# ax.set_ylim(1e-10,1e7)

for i in range(len(Sigma_g)):
    ax.loglog(
        R, Sigma_g[i],
        color="red", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma_d[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_g))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_g[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_g))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_g)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_g$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_g[frame_indices[0]], color="red", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_g[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma D

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_d))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_D (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e3)

for i in range(len(Sigma_d)):
    ax.loglog(
        R, Sigma_d[i],
        color="red", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = "--"
    )
    # ax.loglog(
    #     R, Sigma_d[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_d))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_d[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_d))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_d)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e3)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_d$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_d[frame_indices[0]], color="red", lw=1.2, ls = "--")
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_d[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma Total

In [ ]:
Sigma = Sigma_g + Sigma_d + Sigma_peb + Sigma_pmal

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(Sigma)):
    ax.loglog(
        R, Sigma[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Sigma Pebbles

In [ ]:
Sigma_bs = Sigma_peb

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(Sigma_bs))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Sigma_bs (g cm^-2)')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(Sigma_bs)):
    ax.loglog(
        R, Sigma_bs[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, Sigma_bs[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(Sigma_bs))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(Sigma_bs[:n_snap])

    sigma_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, sigma_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(Sigma_bs))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_sigma = np.asarray(Sigma_bs)
positive_sigma = positive_sigma[np.isfinite(positive_sigma) & (positive_sigma > 0)]
# ax.set_ylim(positive_sigma.min() * 0.8, positive_sigma.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"$\Sigma_bs$ (g cm$^{-2}$)")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, Sigma_bs[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(Sigma_bs[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        sigma = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, sigma]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

## Stokes Number

In [ ]:
St = St_peb

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi = 300)
alphas = np.array(np.linspace(0.15, 1.0, len(St))) # ticks on all four sides
ax.tick_params(axis="both", which="major", direction="in", length=6, width=1.1, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8, top=True, right=True)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which="major", alpha=0.18, lw=0.8)
ax.set_xlabel('R (AU)')
ax.set_ylabel('Stokes Number')
#ax.set_xlim(8,10)
ax.set_ylim(1e-10,1e6)

for i in range(len(St)):
    ax.loglog(
        R, St[i],
        color="blue", alpha=alphas[i],
        label=rf"t={tsnap[i]:.3f} Myr", lw=1.0
    )
    # ax.loglog(
    #     R, St[i],
    #     color="red", alpha=alphas[i],
    #     label=rf"t={tsnap[i]:.3f} Myr", lw=1.0, ls = 'dashed'
    # )
    rdz_at_snap = np.interp(tsnap[i], np.asarray(t) / 1e6, Rdz)
    ax.axvline(rdz_at_snap, ymax=0.1, color = 'black', ls = 'dotted', alpha = alphas[i])
    ax.axvline(rdz_at_snap, ymin=0.9, color = 'black', ls = 'dotted', alpha = alphas[i])

# Planet markers
n_snap = min(len(tsnap), len(St))
time_myr = np.asarray(t) / 1e6

all_planet_masses = np.concatenate([
    np.asarray(Mtot[pid]) for pid in planet_ids
])
all_planet_masses = all_planet_masses[
    np.isfinite(all_planet_masses) & (all_planet_masses > 0)
]

log_mass_min = np.log10(all_planet_masses.min())
log_mass_span = np.log10(all_planet_masses.max()) - log_mass_min

for p_index, pid in enumerate(planet_ids[:]):
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]

    planet_r = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Rp[pid][:n_track])
    )
    planet_mass = np.interp(
        tsnap[:n_snap], track_time, np.asarray(Mtot[pid][:n_track])
    )

    # Only mark planets within the plotted radial grid.
    radii = planet_r
    profiles = np.asarray(St[:n_snap])

    St_at_planet = [
        np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        for radius, profile in zip(radii, profiles)
    ]

    log_mass = np.log10(np.maximum(planet_mass, all_planet_masses.min()))
    marker_sizes = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)

    ax.scatter(
        radii, St_at_planet,
        s=marker_sizes,
        color=f"C{p_index % 10}",
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=f"Planet {pid}",
        zorder=5,
    )

ax.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.05, 1.0), loc='upper left')

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

n_snap = min(len(tsnap), len(St))
snapshot_times = np.asarray(tsnap[:n_snap])
time_myr = np.asarray(t) / 1e6

# Use None for every snapshot, or specify times in Myr.
requested_times_myr = None
if requested_times_myr is None:
    frame_indices = np.arange(n_snap)
else:
    frame_indices = np.unique([
        np.abs(snapshot_times - target).argmin()
        for target in requested_times_myr
    ])

# Interpolate planet tracks onto the gas-profile snapshot times.
planet_tracks = {}
for pid in planet_ids:
    n_track = min(len(t), len(Rp[pid]), len(Mtot[pid]))
    track_time = time_myr[:n_track]
    planet_tracks[pid] = (
        np.interp(snapshot_times, track_time, np.asarray(Rp[pid][:n_track])),
        np.interp(snapshot_times, track_time, np.asarray(Mtot[pid][:n_track])),
    )

all_masses = np.concatenate([mass for _, mass in planet_tracks.values()])
all_masses = all_masses[np.isfinite(all_masses) & (all_masses > 0)]
log_mass_min = np.log10(all_masses.min())
log_mass_span = np.log10(all_masses.max()) - log_mass_min

fig, ax = plt.subplots(figsize=(8, 4), dpi=250)
ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_xlim(R[0], R[-1])

positive_St = np.asarray(St)
positive_St = positive_St[np.isfinite(positive_St) & (positive_St > 0)]
# ax.set_ylim(positive_St.min() * 0.8, positive_St.max() * 1.2)
ax.set_ylim(1e-10,1e6)

ax.set_xlabel("R (AU)")
ax.set_ylabel(r"Stokes Number")
ax.grid(True, which="major", alpha=0.18)

profile_line, = ax.loglog(R, St[frame_indices[0]], color="blue", lw=1.2)
deadzone_low = ax.axvline(Rdz[0], ymax=0.1, color="black", ls="dotted")
deadzone_high = ax.axvline(Rdz[0], ymin=0.9, color="black", ls="dotted")

planet_markers = {}
for p_index, pid in enumerate(planet_ids):
    planet_markers[pid] = ax.scatter(
        [], [], color=f"C{p_index % 10}", edgecolors="black",
        linewidths=0.4, label=f"Planet {pid}", zorder=5
    )

ax.legend(frameon=False, fontsize=6, loc="best")
title = ax.set_title("")

def update(frame):
    profile = np.asarray(St[frame])
    profile_line.set_ydata(profile)

    r_dz = np.interp(snapshot_times[frame], time_myr, Rdz)
    deadzone_low.set_xdata([r_dz, r_dz])
    deadzone_high.set_xdata([r_dz, r_dz])

    for pid, marker in planet_markers.items():
        planet_r, planet_mass = planet_tracks[pid]
        radius = planet_r[frame]
        mass = planet_mass[frame]

        St_at_planet = np.exp(np.interp(np.log(radius), np.log(R), np.log(profile)))
        log_mass = np.log10(max(mass, all_masses.min()))
        size = 5 + 145 * (log_mass - log_mass_min) / max(log_mass_span, 1e-12)
        marker.set_offsets([[radius, St_at_planet]])
        marker.set_sizes([size])

    title.set_text(f"t = {snapshot_times[frame]:.3f} Myr")
    return profile_line, deadzone_low, deadzone_high, title, *planet_markers.values()

animation = FuncAnimation(
    fig, update, frames=frame_indices, interval=150, blit=False,
)
plt.close(fig)
HTML(animation.to_jshtml())

# Combined plots 

## 0.1 Myr DZ + noDZ 1e-5 (-> 1e-3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from IPython.display import HTML

# Change these two labels to compare another pair of runs (e.g. the 3Myr pair).
no_dz_label = "0.1Myr_NoDZ_1e-8_0.1Msol_50Rd_1e-5"
dz_label = "0.1Myr_DZ_1e-8_0.1Msol_50Rd_1e-5_1e-3"


def combined_growth_animation(no_dz_data, dz_data, cadence_myr=0.01):
    # Keep data and artists local so later notebook cells cannot change the animation.
    def prepare(data, has_deadzone=False):
        ids = sorted(data["Rp"], key=int)
        if not ids:
            raise ValueError("The selected run contains no planet tracks.")
        lengths = [len(data["t"])]
        for pid in ids:
            lengths.extend(len(data[key][pid]) for key in ("Rp", "Mcs", "Mes"))
        if has_deadzone:
            lengths.append(len(data["R_dz_t"]))
        n = min(lengths)
        if n == 0:
            raise ValueError("The selected run contains an empty time series.")
        times = np.asarray(data["t"][:n]) / 1e6
        if not np.all(np.isfinite(times)) or np.any(np.diff(times) < 0):
            raise ValueError("Simulation times must be finite and nondecreasing.")
        tracks = []
        for pid in ids:
            radius = np.asarray(data["Rp"][pid][:n])
            envelope = np.asarray(data["Mes"][pid][:n])
            mass = np.asarray(data["Mcs"][pid][:n]) + envelope
            valid = (np.isfinite(radius) & (radius > 0)
                     & np.isfinite(mass) & (mass > 0) & np.isfinite(envelope))
            fraction = np.divide(envelope, mass, out=np.zeros(n), where=valid)
            tracks.append((radius, mass, fraction, valid))
        return times, tracks

    if not np.isfinite(cadence_myr) or cadence_myr <= 0:
        raise ValueError("cadence_myr must be positive and finite.")
    no_times, no_tracks = prepare(no_dz_data)
    dz_times, dz_tracks = prepare(dz_data, has_deadzone=True)
    deadzone = np.asarray(dz_data["R_dz_t"][:len(dz_times)])

    # Compare the shared time interval; never show a sample from the future.
    start = max(no_times[0], dz_times[0])
    end = min(no_times[-1], dz_times[-1])
    if end < start:
        raise ValueError("The selected simulations have no overlapping time interval.")
    frame_times = np.unique(np.append(np.arange(start, end, cadence_myr), end))

    fig, (ax, no_cax, dz_cax) = plt.subplots(
        1, 3, figsize=(10, 6), dpi=150, layout="constrained",
        gridspec_kw={"width_ratios": [1, 0.035, 0.035]},
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    for coordinate, setter in ((0, ax.set_xlim), (1, ax.set_ylim)):
        values = np.concatenate([
            track[coordinate][track[3]] for track in no_tracks + dz_tracks
        ])
        if not values.size:
            plt.close(fig)
            raise ValueError("No finite, positive track points are available to plot.")
        setter(values.min() * 0.8, values.max() * 1.2)
    ax.set_xlabel("Distance (AU)")
    ax.set_ylabel("Total mass")
    ax.grid(True, which="both", alpha=0.2)
    ax.set_axisbelow(True)

    norm = Normalize(vmin=0.0, vmax=1.0)
    # Exclude the pale end of Oranges so low envelope fractions remain visible.
    no_cmap = LinearSegmentedColormap.from_list(
        "no_deadzone_oranges", plt.cm.Oranges(np.linspace(0.45, 1.0, 256)),
    )
    dz_cmap = plt.cm.viridis
    style = dict(c=[], norm=norm, s=35, edgecolors="none")
    no_scatter = ax.scatter(
        [], [], cmap=no_cmap, marker="o", alpha=0.65, zorder=1, **style,
    )
    dz_scatter = ax.scatter(
        [], [], cmap=dz_cmap, marker="o", alpha=0.7, zorder=2, **style,
    )
    deadzone_line = ax.axvline(
        deadzone[0], ymin=0, ymax=0.10, color="red",
        linestyle=":", linewidth=1.5, zorder=3,
    )
    for cmap, label, cax in (
        (no_cmap, "No deadzone", no_cax), (dz_cmap, "Deadzone", dz_cax),
    ):
        colorbar = fig.colorbar(
            plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax,
        )
        colorbar.set_label(label + "\n" + r"$M_{\rm env}/M_{\rm tot}$")
    ax.legend(handles=[
        Line2D([], [], marker="o", linestyle="", color=no_cmap(0.4), alpha=0.65,
               label="No deadzone (underneath)"),
        Line2D([], [], marker="o", linestyle="", color=dz_cmap(0.4), alpha=0.7,
               label="Deadzone"),
        Line2D([], [], color="red", linestyle=":", label="Deadzone edge"),
    ], loc="best", fontsize=8)
    title = ax.set_title("")

    def update_trails(scatter, times, tracks, current_time):
        stop = np.searchsorted(times, current_time, side="right")
        offsets, fractions = [], []
        for radius, mass, fraction, valid in tracks:
            mask = valid[:stop]
            offsets.append(np.column_stack((radius[:stop][mask], mass[:stop][mask])))
            fractions.append(fraction[:stop][mask])
        scatter.set_offsets(np.concatenate(offsets))
        scatter.set_array(np.concatenate(fractions))

    def update(current_time):
        update_trails(no_scatter, no_times, no_tracks, current_time)
        update_trails(dz_scatter, dz_times, dz_tracks, current_time)
        dz_index = np.searchsorted(dz_times, current_time, side="right") - 1
        radius = deadzone[dz_index]
        deadzone_line.set_visible(bool(np.isfinite(radius) and radius > 0))
        deadzone_line.set_xdata([radius, radius])
        title.set_text(rf"$t={current_time:.3f}$ Myr")
        return no_scatter, dz_scatter, deadzone_line, title

    animation = FuncAnimation(fig, update, frames=frame_times, interval=150, blit=False)
    return fig, animation


combined_fig, combined_animation = combined_growth_animation(
    datasets[no_dz_label]["data"], datasets[dz_label]["data"], cadence_myr=0.01,
)
# Optional GIF export (relative to the notebook's working directory):
# combined_animation.save("combined_growth_tracks.gif", writer=PillowWriter(fps=8), dpi=150)
plt.close(combined_fig)
HTML(combined_animation.to_jshtml())


## 3.0 Myr DZ + noDZ 1e-5 (-> 1e-3)

In [ ]:
no_dz_label = "3Myr_NoDZ_1e-8_0.1Msol_50Rd_1e-5"
dz_label = "3Myr_DZ_1e-8_0.1Msol_50Rd_1e-5_1e-3"

combined_fig, combined_animation = combined_growth_animation(
    datasets[no_dz_label]["data"], datasets[dz_label]["data"], cadence_myr=0.05,
)
# Optional GIF export (relative to the notebook's working directory):
# combined_animation.save("combined_growth_tracks.gif", writer=PillowWriter(fps=8), dpi=150)
plt.close(combined_fig)
HTML(combined_animation.to_jshtml())